<a href="https://colab.research.google.com/github/Olamyy/tensorcas/blob/hash-cache-no-op-path/notebooks/sklearn_xgboost_dvc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# tensorcas vs joblib / DVC — sklearn and XGBoost

3-way storage comparison for tree model checkpoint sequences.

**Methods compared:**

| Method | What it does |
|---|---|
| joblib / pickle | Naive full save — serializes the complete model every step |
| DVC (real) | File-level versioning — tracks each checkpoint file with `dvc add` + `dvc push` to a local remote |
| tensorcas | Tensor-level dedup — extracts per-tree arrays, hashes each, writes only new or changed tensors |

Note: safetensors is not included — it is a tensor format for neural networks and does not apply to tree models.

**Scenarios:**

1. **sklearn warm-start** — 20 checkpoints, 50 trees/step (10–1,000 trees total)
2. **XGBoost warm-start** — 20 checkpoints, 50 rounds/step (50–1,000 rounds total)
3. **sklearn hyperparameter sweep** — 5 runs from the same base, different `max_depth` / `learning_rate`
4. **XGBoost hyperparameter sweep** — 5 runs, different `eta` / `max_depth`

**DVC setup:** local remote at a temp directory. Each checkpoint tracked with `dvc add` + `dvc push`.
**tensorcas setup:** shared `TensorCasStore` root per scenario. Bytes = sum of `.chunk` files under `objects/`.
Both DVC and tensorcas use zstd level 3 compression.

In [ ]:
!pip install -q git+https://github.com/Olamyy/tensorcas.git@hash-cache-no-op-path zstandard scikit-learn xgboost dvc joblib

In [ ]:
import copy
import joblib
import pickle
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import numpy as np
import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier

sys.path.insert(0, str(Path(".").resolve()))
from utils import tensorcas_bytes, _fmt_bytes

from tensorcas.store import TensorCasStore
from tensorcas.adapters.sklearn import SklearnAdapter
from tensorcas.adapters.xgboost import XGBoostAdapter

print("Imports OK")

## Shared helpers

In [ ]:
def dvc_init(repo_dir: Path, remote_dir: Path) -> None:
    repo_dir.mkdir(parents=True, exist_ok=True)
    remote_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", "bench@tensorcas"], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.name", "tensorcas Bench"], cwd=repo_dir, check=True)
    subprocess.run(["dvc", "init", "-q"], cwd=repo_dir, check=True)
    subprocess.run(
        ["dvc", "remote", "add", "-d", "local", str(remote_dir)],
        cwd=repo_dir, check=True,
    )


def dvc_track_and_push(repo_dir: Path, checkpoint_path: Path) -> None:
    subprocess.run(["dvc", "add", str(checkpoint_path)], cwd=repo_dir, check=True,
                   capture_output=True)
    subprocess.run(["dvc", "push"], cwd=repo_dir, check=True, capture_output=True)


def dvc_cache_bytes(remote_dir: Path) -> int:
    return sum(p.stat().st_size for p in remote_dir.rglob("*") if p.is_file())


def raw_bytes(files) -> int:
    return sum(p.stat().st_size for p in files)


def print_comparison(
    label: str,
    n_checkpoints: int,
    raw_b: int,
    joblib_b: int,
    dvc_b: int,
    tensorcas_b: int,
) -> None:
    def _savings(baseline, b):
        return (baseline - b) / baseline * 100 if baseline else 0

    print(f"\n{label}")
    print(f"  Checkpoints : {n_checkpoints}")
    print(f"  Raw total   : {_fmt_bytes(raw_b)}")
    print(f"  joblib      : {_fmt_bytes(joblib_b)}   (full save, no dedup)")
    print(f"  DVC         : {_fmt_bytes(dvc_b)}   (file-level dedup, {_savings(joblib_b, dvc_b):.1f}% vs joblib)")
    print(f"  tensorcas        : {_fmt_bytes(tensorcas_b)}   (tensor-level dedup, {_savings(joblib_b, tensorcas_b):.1f}% vs joblib, {_savings(dvc_b, tensorcas_b):.1f}% vs DVC)")


print("Helpers OK")

## sklearn helpers

In [ ]:
def make_dataset(seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((2000, 20)).astype(np.float32)
    y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(int)
    return X, y


def train_sklearn(
    seed: int = 0,
    n_steps: int = 20,
    trees_per_step: int = 50,
    max_depth: int = 3,
    learning_rate: float = 0.1,
):
    X, y = make_dataset(seed)
    model = GradientBoostingClassifier(
        n_estimators=trees_per_step,
        max_depth=max_depth,
        learning_rate=learning_rate,
        warm_start=True,
        random_state=seed,
    )
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models


print("sklearn helpers OK")

## XGBoost helpers

In [ ]:
def train_xgboost(
    seed: int = 0,
    n_steps: int = 20,
    rounds_per_step: int = 50,
    max_depth: int = 3,
    eta: float = 0.1,
):
    X, y = make_dataset(seed)
    dtrain = xgb.DMatrix(X, label=y)
    params = {
        "objective": "binary:logistic",
        "max_depth": max_depth,
        "eta": eta,
        "seed": seed,
        "verbosity": 0,
    }
    boosters = []
    booster = None
    for step in range(n_steps):
        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=rounds_per_step,
            xgb_model=booster,
            verbose_eval=False,
        )
        boosters.append(booster)
    return boosters


print("XGBoost helpers OK")

---
## Scenario 1 — sklearn warm-start (20 steps, 50 trees/step)

Single run: 20 checkpoints, growing from 50 to 1,000 trees.

**Expected:** joblib stores 20 growing pickle files (full model each time).
DVC tracks those same files with file-level dedup — each file is unique
(the model grows each step), so DVC saves nothing over raw joblib.
tensorcas extracts per-tree arrays and only writes new trees each step.
After step 1, no existing tree ever changes — all savings come from the no-op path.

In [ ]:
print("Training sklearn warm-start (20 steps × 50 trees)...", end=" ", flush=True)
t0 = time.time()
sklearn_models = train_sklearn(seed=0)
print(f"{time.time() - t0:.1f}s")
print(f"Final model: {sklearn_models[-1].n_estimators} trees")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    ckpt_dir   = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    joblib_files = []
    for step, model in enumerate(sklearn_models, 1):
        path = ckpt_dir / f"step_{step:02d}.pkl"
        joblib.dump(model, path)
        joblib_files.append(path)
        dvc_track_and_push(dvc_repo, path)

    with TensorCasStore(root=tensorcas_root, run_id="sklearn_warmstart", adapter=SklearnAdapter()) as store:
        for step, model in enumerate(sklearn_models, 1):
            store.save(model, step=step)

    s1_raw_b    = raw_bytes(joblib_files)
    s1_joblib_b = s1_raw_b
    s1_dvc_b    = dvc_cache_bytes(dvc_remote)
    s1_tc_b     = tensorcas_bytes(tensorcas_root)

print_comparison(
    "Scenario 1 — sklearn warm-start (20 steps × 50 trees/step)",
    len(sklearn_models), s1_raw_b, s1_joblib_b, s1_dvc_b, s1_tc_b,
)

---
## Scenario 2 — XGBoost warm-start (20 steps, 50 rounds/step)

Single run: 20 checkpoints, growing from 50 to 1,000 rounds.

**Expected:** Similar to sklearn. Each `.ubj` checkpoint grows by ~50 rounds.
DVC stores all 20 unique files. tensorcas writes each tree once; only the
`__skeleton__` (model metadata) changes every step.

In [ ]:
print("Training XGBoost warm-start (20 steps × 50 rounds)...", end=" ", flush=True)
t0 = time.time()
xgb_models = train_xgboost(seed=0)
print(f"{time.time() - t0:.1f}s")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    ckpt_dir   = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    xgb_files = []
    for step, booster in enumerate(xgb_models, 1):
        path = ckpt_dir / f"step_{step:02d}.ubj"
        booster.save_model(str(path))
        xgb_files.append(path)
        dvc_track_and_push(dvc_repo, path)

    with TensorCasStore(root=tensorcas_root, run_id="xgb_warmstart", adapter=XGBoostAdapter()) as store:
        for step, booster in enumerate(xgb_models, 1):
            store.save(booster, step=step)

    s2_raw_b    = raw_bytes(xgb_files)
    s2_joblib_b = s2_raw_b
    s2_dvc_b    = dvc_cache_bytes(dvc_remote)
    s2_tc_b     = tensorcas_bytes(tensorcas_root)

print_comparison(
    "Scenario 2 — XGBoost warm-start (20 steps × 50 rounds/step)",
    len(xgb_models), s2_raw_b, s2_joblib_b, s2_dvc_b, s2_tc_b,
)

---
## Scenario 3 — sklearn hyperparameter sweep (5 runs)

5 independent runs from the same dataset (seed=0), varying `max_depth`
and `learning_rate`. Each run produces 20 checkpoints.

**Expected:** DVC stores all 5 × 20 = 100 checkpoint files independently
— no cross-run awareness. tensorcas shares a single store across all runs;
trees that happen to be identical across runs (same depth, same splits)
are written once. sklearn trees are deterministic given the same data and
hyperparameters, so runs with identical `max_depth` / `lr` would share
all trees — but this grid uses distinct configs, so sharing is limited.

In [ ]:
SKLEARN_HP_GRID = [
    {"run_id": "depth2_lr005",  "max_depth": 2, "learning_rate": 0.05},
    {"run_id": "depth3_lr010",  "max_depth": 3, "learning_rate": 0.10},
    {"run_id": "depth4_lr010",  "max_depth": 4, "learning_rate": 0.10},
    {"run_id": "depth3_lr020",  "max_depth": 3, "learning_rate": 0.20},
    {"run_id": "depth5_lr005",  "max_depth": 5, "learning_rate": 0.05},
]

sklearn_hp_models = {}
for cfg in SKLEARN_HP_GRID:
    t0 = time.time()
    print(f"  {cfg['run_id']}...", end=" ", flush=True)
    sklearn_hp_models[cfg["run_id"]] = train_sklearn(
        seed=0, max_depth=cfg["max_depth"], learning_rate=cfg["learning_rate"]
    )
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints: {sum(len(v) for v in sklearn_hp_models.values())}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    ckpt_dir   = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    all_files = []
    s3_tensorcas_size_after = {}
    n_checkpoints = 0

    for cfg in SKLEARN_HP_GRID:
        run_id = cfg["run_id"]
        run_dir = ckpt_dir / run_id
        run_dir.mkdir()

        for step, model in enumerate(sklearn_hp_models[run_id], 1):
            path = run_dir / f"step_{step:02d}.pkl"
            joblib.dump(model, path)
            all_files.append(path)
            dvc_track_and_push(dvc_repo, path)
            n_checkpoints += 1

        with TensorCasStore(root=tensorcas_root, run_id=run_id, adapter=SklearnAdapter()) as store:
            for step, model in enumerate(sklearn_hp_models[run_id], 1):
                store.save(model, step=step)

        s3_tensorcas_size_after[run_id] = tensorcas_bytes(tensorcas_root)
        print(f"  {run_id}: done")

    s3_raw_b    = raw_bytes(all_files)
    s3_joblib_b = s3_raw_b
    s3_dvc_b    = dvc_cache_bytes(dvc_remote)
    s3_tc_b     = tensorcas_bytes(tensorcas_root)

print_comparison(
    "Scenario 3 — sklearn HP sweep (5 runs × 20 steps)",
    n_checkpoints, s3_raw_b, s3_joblib_b, s3_dvc_b, s3_tc_b,
)

print("\ntensorcas store size after each run:")
prev = 0
for run_id, size in s3_tensorcas_size_after.items():
    print(f"  {run_id:<20} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

---
## Scenario 4 — XGBoost hyperparameter sweep (5 runs)

5 independent XGBoost runs from the same dataset, varying `eta` and `max_depth`.
Each run produces 20 checkpoints. Same shared tensorcas store across all runs.

**Expected:** DVC stores 100 independent `.ubj` files. tensorcas shares individual
tree tensors that happen to be identical across runs. Unlike sklearn, XGBoost
tree structure is less likely to produce identical tensors across different
`eta` values — cross-run savings will be modest.

In [ ]:
XGB_HP_GRID = [
    {"run_id": "depth3_eta010", "max_depth": 3, "eta": 0.10},
    {"run_id": "depth3_eta020", "max_depth": 3, "eta": 0.20},
    {"run_id": "depth4_eta010", "max_depth": 4, "eta": 0.10},
    {"run_id": "depth4_eta005", "max_depth": 4, "eta": 0.05},
    {"run_id": "depth5_eta010", "max_depth": 5, "eta": 0.10},
]

xgb_hp_models = {}
for cfg in XGB_HP_GRID:
    t0 = time.time()
    print(f"  {cfg['run_id']}...", end=" ", flush=True)
    xgb_hp_models[cfg["run_id"]] = train_xgboost(
        seed=0, max_depth=cfg["max_depth"], eta=cfg["eta"]
    )
    print(f"{time.time() - t0:.1f}s")

print(f"\nTotal checkpoints: {sum(len(v) for v in xgb_hp_models.values())}")

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    dvc_repo   = tmp / "dvc_repo"
    dvc_remote = tmp / "dvc_remote"
    tensorcas_root  = tmp / "tensorcas"
    ckpt_dir   = dvc_repo / "checkpoints"

    dvc_init(dvc_repo, dvc_remote)
    ckpt_dir.mkdir()

    all_files = []
    s4_tensorcas_size_after = {}
    n_checkpoints = 0

    for cfg in XGB_HP_GRID:
        run_id = cfg["run_id"]
        run_dir = ckpt_dir / run_id
        run_dir.mkdir()

        for step, booster in enumerate(xgb_hp_models[run_id], 1):
            path = run_dir / f"step_{step:02d}.ubj"
            booster.save_model(str(path))
            all_files.append(path)
            dvc_track_and_push(dvc_repo, path)
            n_checkpoints += 1

        with TensorCasStore(root=tensorcas_root, run_id=run_id, adapter=XGBoostAdapter()) as store:
            for step, booster in enumerate(xgb_hp_models[run_id], 1):
                store.save(booster, step=step)

        s4_tensorcas_size_after[run_id] = tensorcas_bytes(tensorcas_root)
        print(f"  {run_id}: done")

    s4_raw_b    = raw_bytes(all_files)
    s4_joblib_b = s4_raw_b
    s4_dvc_b    = dvc_cache_bytes(dvc_remote)
    s4_tc_b     = tensorcas_bytes(tensorcas_root)

print_comparison(
    "Scenario 4 — XGBoost HP sweep (5 runs × 20 steps)",
    n_checkpoints, s4_raw_b, s4_joblib_b, s4_dvc_b, s4_tc_b,
)

print("\ntensorcas store size after each run:")
prev = 0
for run_id, size in s4_tensorcas_size_after.items():
    print(f"  {run_id:<20} {_fmt_bytes(size):>10}  (+{_fmt_bytes(size - prev)})")
    prev = size

---
## Summary

| Scenario | Checkpoints | joblib | DVC | tensorcas | tensorcas vs joblib | tensorcas vs DVC |
|---|---|---|---|---|---|---|
| 1 — sklearn warm-start (1 run × 20 steps) | 20 | 12.5 MB | 12.5 MB | 216.5 KB | **98.3%** | **98.3%** |
| 2 — XGBoost warm-start (1 run × 20 steps) | 20 | 9.8 MB | 9.8 MB | 448.1 KB | **95.5%** | **95.5%** |
| 3 — sklearn HP sweep (5 runs × 20 steps) | 100 | 90.9 MB | 90.9 MB | 1.5 MB | **98.4%** | **98.4%** |
| 4 — XGBoost HP sweep (5 runs × 20 steps) | 100 | 50.7 MB | 50.7 MB | 2.2 MB | **95.7%** | **95.7%** |

**Why DVC saves nothing over joblib for tree models:**
Each warm-start checkpoint is a unique file (the model grows each step).
DVC's file-level dedup only fires when two checkpoint files are byte-identical —
which never happens in a warm-start sequence. DVC ≈ joblib in all 4 scenarios.

**Why sklearn savings (98.3–98.4%) are higher than XGBoost (95.5–95.7%):**
sklearn stores per-tree arrays as separate tensors — once written, a tree is never
touched again. XGBoost's `__skeleton__` (model metadata JSON with round count and
statistics) changes on every step as the model grows — this is the sole source of
extra writes. Individual XGBoost tree tensors are 100% stable after first write.

**sklearn HP sweep — per-run marginal cost grows with max_depth:**
Deeper trees produce larger, more distinct arrays. depth2 costs 97 KB; depth5
costs 625 KB. No cross-run sharing because different hyperparameters produce
different tree structures.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 150})

from pathlib import Path
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

MB = 1024**2

# --- Figure 1: Storage comparison — 3 methods × 4 scenarios ---
# Read from s1_*, s2_*, s3_*, s4_* variables computed in the scenario cells above
scenarios = [
    "sklearn\nwarm-start\n(1×20 steps)",
    "XGBoost\nwarm-start\n(1×20 steps)",
    "sklearn\nHP sweep\n(5×20 steps)",
    "XGBoost\nHP sweep\n(5×20 steps)",
]
joblib_mb    = [s1_joblib_b / MB, s2_joblib_b / MB, s3_joblib_b / MB, s4_joblib_b / MB]
dvc_mb       = [s1_dvc_b / MB,    s2_dvc_b / MB,    s3_dvc_b / MB,    s4_dvc_b / MB]
tensorcas_mb = [s1_tc_b / MB,     s2_tc_b / MB,     s3_tc_b / MB,     s4_tc_b / MB]

x = range(len(scenarios))
width = 0.28
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width for i in x], joblib_mb,    width, label="joblib/pickle", color="#4C72B0")
ax.bar([i         for i in x], dvc_mb,       width, label="DVC",           color="#DD8452")
ax.bar([i + width for i in x], tensorcas_mb, width, label="TensorCas",     color="#55A868")
ax.set_xticks(list(x))
ax.set_xticklabels(scenarios)
ax.set_ylabel("Total storage (MB)")
ax.set_title("Storage comparison: joblib vs DVC vs TensorCas\n(sklearn and XGBoost warm-start + HP sweep)")
ax.legend()
ax.set_yscale("log")
ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda v, _: f"{v:g} MB"))
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sklearn_xgboost_storage_comparison.png")
plt.show()
print("Saved figures/sklearn_xgboost_storage_comparison.png")

# --- Figure 2: sklearn HP sweep — cumulative tensorcas cost per run ---
# Read from s3_tensorcas_size_after collected in scenario 3 cell
sklearn_configs = list(s3_tensorcas_size_after.keys())
cumulative_mb_vals = [v / MB for v in s3_tensorcas_size_after.values()]
prev_vals = [0] + list(s3_tensorcas_size_after.values())[:-1]
marginal_vals_kb = [(cur - prev) / 1024 for cur, prev in zip(s3_tensorcas_size_after.values(), prev_vals)]

# Shorten config labels for readability
short_labels = [k.replace("depth", "d").replace("_lr", "\nlr=") for k in sklearn_configs]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(short_labels, cumulative_mb_vals, color="#55A868", width=0.5)
ax.set_ylabel("Cumulative TensorCas store size (MB)")
ax.set_title("sklearn HP sweep — cumulative store size per run\n(incremental cost grows with max_depth)")
for i, (cum, mg) in enumerate(zip(cumulative_mb_vals, marginal_vals_kb)):
    ax.text(i, cum + max(cumulative_mb_vals) * 0.02, f"+{mg:.0f} KB", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "sklearn_hp_marginal_cost.png")
plt.show()
print("Saved figures/sklearn_hp_marginal_cost.png")